# Projeto final INF <numero-disciplina>

Neste projeto final iremos criar um classificador de insetos para tentar obter a espécie ou pelo menos o gênero de uma instância.

Vamos utilizar datasets coletados manualmente e datasets coletados via Kaggle e Zenodo. 

Datasets:
- João
  - 454 imagens
  - 124 classes
- [Insect Identification in the Wild: The AMI Dataset](https://zenodo.org/records/12554005)
  - expert-annotated dataset of 2,893 insect camera trap images (representing 52,948 labeled insects) collected from a global network of automated camera traps, designed to test in-the-wild performance.
- [InsectDCT: Datasets for training and evaluating pipeline processing insect camera recordings](https://zenodo.org/records/21154490?preview_file=classifierTrainV6.zip)
  - 90.591 imagens
  - 207 classes
- [Dangerous Farm Insects Dataset](https://www.kaggle.com/datasets/tarundalal/dangerous-insects-dataset)
  - 1.606 imagens
  - 15 classes
- [Insect Images With Scientific Names](https://www.kaggle.com/datasets/shameinew/insect-images-with-scientific-names)
  - 122.665 imagens
  - 2.273 classes
- [IP102-Dataset](https://www.kaggle.com/datasets/rtlmhjbn/ip02-dataset)
  - 75.222 imagens
  - 102 classes
- [Insect identification from habitus images](https://www.kaggle.com/datasets/kmldas/insect-identification-from-habitus-images)
  - 63.655 imagens
  - 291 classes

Total de imagens sem filtrar/mesclar: 407.141
Total de classes sem filtrar/mesclar: 3.012

Em um primeiro momento iremos padronizar os nomes de cada inseto com o padrão BIN do BOLD Data Portal. Em seguida vamos unificar as classes e enriquecer as classes obtidas com as imagens do BOLD Data Portal.

Posteriormente vamos filtrar as classes com a seguinte regra:
- classes obtidas pelo João entram (independente da quantidade de imagens).
- classes que não são latino americanas.
- classes com menos imagens saem (pensar em algo como tirar as classes com menos de dois desvios padrão da quantidade de imagens entre as classes).

Total de imagens obtidas:
Total de classes obtidas:
Média de imagens por classe:
Mediana de imagens por classe:
Desvio padrão de imagens por classe:
Min:
Max:


Com as classes fechadas, vamos tratar os dados. Temos 3 possíveis problemas: 
- fotos por microscópio tem cerca de 40% da imagem da tela preta (o que está fora do foco). Solução: algorítmo que corta a imagem (o mesmo deve estar presente no pipeline)
- fotos com insetos em folhas nos dados de treino (cor pode esconder o inseto)
- algumas fotos são grandes, necessário reduzir o tamanho
  
Tratados os problemas, vamos pensar nos algorítmos para as classificações. Tenho em mente 3 abordagens:
- Aplicar no Yolo e fazer fine tunning
- Aplicar em modelos pré-treinados do hugging-face e fazer fine tunning
- Hierarchical Mixture of Experts

A depender do tempo, testar o máximo de alternativas possíveis e compará-las.

In [43]:
#imports
import requests
import os 
from typing import Any, Dict, List, Optional, Union
import json
import re
import shutil

In [87]:
# endpoints importantes:

# realiza request
def realiza_request(
    url: str
) -> Optional[Union[Dict[str, Any], List[Any]]]:
    """Realiza uma requisição HTTP GET solicitando payload em formato JSON.

    Args:
        url (str): Endereço do recurso HTTP/HTTPS a ser consultado.

    Returns:
        Optional[Union[Dict[str, Any], List[Any]]]: Conteúdo JSON desserializado
            em caso de sucesso (status 2xx), ou None se houver falha.
    """
    headers = {"accept": "application/json"}
    error_map = {
                400: "Bad Request: Parâmetros ou estrutura de requisição inválidos.",
                401: "Unauthorized: Autenticação necessária.",
                403: "Forbidden: Acesso negado ao recurso.",
                404: "Not Found: O recurso solicitado não existe.",
                500: "Internal Server Error: Erro interno no servidor.",
                502: "Bad Gateway: Resposta inválida do servidor upstream.",
                503: "Service Unavailable: Serviço temporariamente indisponível.",
            }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.HTTPError as http_err:
        status_code = response.status_code
        message = error_map.get(
            status_code, f"Erro HTTP não mapeado ({status_code}): {http_err}"
        )
        print("Falha de requisição para '%s': %s", url, message)

    except requests.exceptions.Timeout:
        print("Tempo limite de requisição excedido para '%s'.", url)

    except requests.exceptions.ConnectionError:
        print("Falha na conexão de rede ao tentar acessar '%s'.", url)

    except requests.exceptions.RequestException as err:
        print("Erro inesperado na comunicação com '%s': %s", url, err)

    return None

# gemini me ajudou aqui, achei chique
def get_lowest_taxonomic_value(data: Dict[str, Optional[str]]) -> Optional[str]:
    """Retorna o valor do nível taxonômico mais específico que não seja nulo.
    Cria uma lista de prioridades ordenados pelo valor do dado e pelo rank"""
    TAXONOMIC_HIERARCHY = ["species", "genus", "tribe", "subfamily", "family"]
    return next(
        (data[rank] for rank in TAXONOMIC_HIERARCHY if data.get(rank)),
        None
    )

# api/query/preprocessor
def bold_data_preprocessor(
        termo:str
) -> str :
    '''Endpoint para gerar o triplet_token a partir de um termo
    Args:
        termo (str): Nome da espécie a ser parseada.
    
    Returns:
        str: token parseado para obter o query_id
    '''

    url = f'https://portal.boldsystems.org/api/query/preprocessor?query={termo}'

    response = realiza_request(url)
    triplet_token = response['successful_terms'][0]['matched']

    return triplet_token

# api/query
def bold_data_query(
        triplet_token:str
) -> str :
    '''Endpoint para gerar o query_id a partir do termo
    Args:
        triplet_token (str): token usado para obter o query_id.
    
    Returns:
        str: o query_id
    '''

    url = f'https://portal.boldsystems.org/api/query?query={triplet_token}&extent=limited'

    response = realiza_request(url)
    query_id = response['query_id']

    return query_id

class QueryVazia(Exception):
    """Excessão lançada normalmente quando não é encontrado o nome da espécie"""
    pass

# api/documents/
def bold_data_documents(
        query_id:str
) -> str:
    '''Endpoint que pode retornar o species
    Args:
        query_id (str): id do inseto obtido via bold_data_query().
    
    Returns:
        str: nome da espécie encontrada
    '''

    url = f'https://portal.boldsystems.org/api/documents/{query_id}?length=1&start=0'

    response = realiza_request(url)
    if not response['data']:
        raise QueryVazia("Resposta de documents veio vazio, nome de pasta errado")
    dados = {
        "species":response['data'][0]['species'],
        "genus":response['data'][0]['genus'],
        "tribe":response['data'][0]['tribe'],
        "subfamily":response['data'][0]['subfamily'],
        "family":response['data'][0]['family']
    }

    return get_lowest_taxonomic_value(dados)

# api/images/
def bold_data_images(
        query_id:str
) -> list[str]:
    '''Endpoint que retorna o link para baixar imagens
    Args:
        query_id (str): id do inseto obtido via bold_data_query().
    
    Returns:
        str: lista dos links para download das imagens
    '''
    image_urls = list()
    url = f'https://portal.boldsystems.org/api/images/{query_id}?max_images=-1'

    response = realiza_request(url)
    for image in response['images']:
        image_urls.append(image["image_url"])

    return image_urls



## Tratando arquivos do João

In [77]:
ja_passou = list()

In [114]:
def get_nome_pasta(
        termo:str
) -> str:
    """Função que busca no BALD Data Portal o nome científico dado nome da pasta"""
    termo = re.sub(r'sp\d+', '',termo.lower()).strip().title()
    if ' ' in termo:
        termo = termo.split()[-1].strip()

    # print(f"termo: {termo}")
    triplet_token = bold_data_preprocessor(termo)
    # print(f'triplet_token: {triplet_token}')
    query_id = bold_data_query(triplet_token)
    # print(f'query_id: {query_id}')
    specie = bold_data_documents(query_id)
    # print(f'specie: {specie}')
    return specie

def lista_pastas(
        caminho: str
) -> list[str]:
    """Função que dado um caminho coloca o nome das pastas nele em uma lista"""
    return os.listdir(caminho)

try:

    

    diretorio='/home/bradachi/Downloads/datasets/Insetos taxo'
    diretorio_destino='/home/bradachi/Downloads/datasets/final'
    pastas = lista_pastas(diretorio)
    faltam = len(pastas)
    atual = 1

    for pasta in pastas:
        print(f'[{atual}/{faltam}]')

        atual+=1

        if pasta in ja_passou:
            continue

        nome_novo = get_nome_pasta(pasta)
        origem = f'{diretorio}/{pasta}'
        destino = f'{diretorio_destino}/{nome_novo}'

        shutil.copytree(origem, destino, dirs_exist_ok=True)

        ja_passou.append(pasta)

except Exception as e:
    print(e)
    print(pasta)



[1/123]
[2/123]
[3/123]
[4/123]
[5/123]
[6/123]
[7/123]
[8/123]
[9/123]
[10/123]
[11/123]
[12/123]
[13/123]
[14/123]
[15/123]
[16/123]
[17/123]
[18/123]
[19/123]
[20/123]
[21/123]
[22/123]
[23/123]
[24/123]
[25/123]
[26/123]
[27/123]
[28/123]
[29/123]
[30/123]
[31/123]
[32/123]
[33/123]
[34/123]
[35/123]
[36/123]
[37/123]
[38/123]
[39/123]
[40/123]
[41/123]
[42/123]
[43/123]
[44/123]
[45/123]
[46/123]
[47/123]
[48/123]
[49/123]
[50/123]
[51/123]
[52/123]
[53/123]
[54/123]
[55/123]
[56/123]
[57/123]
[58/123]
[59/123]
[60/123]
[61/123]
[62/123]
[63/123]
[64/123]
[65/123]
[66/123]
[67/123]
[68/123]
[69/123]
[70/123]
[71/123]
[72/123]
[73/123]
[74/123]
[75/123]
[76/123]
[77/123]
[78/123]
[79/123]
[80/123]
[81/123]
[82/123]
[83/123]
[84/123]
[85/123]
[86/123]
[87/123]
[88/123]
[89/123]
[90/123]
[91/123]
[92/123]
[93/123]
[94/123]
[95/123]
[96/123]
[97/123]
[98/123]
[99/123]
[100/123]
[101/123]
[102/123]
[103/123]
[104/123]
[105/123]
[106/123]
[107/123]
[108/123]
[109/123]
[110/123]
[111/123

## Tratando arquivos Dangerous Farm Insects Dataset